# How SDP reads PACE spectra in the Gulf Stream region

Goal: work out where SDP's pigment answers come from in the PACE spectrum, which parts of the spectrum PACE measures in a way the training data never saw, and whether a version of SDP that avoids those parts agrees better with independent chlorophyll measurements.

Data:
- 11,563 PACE pixels inside eddies from the audit sample of 2026-10-13 (`data/sdp_validation/gulf_stream_validity_20261013/envelope/`), about 10 pixels from each of 1,101 eddy composites. A composite is one eddy on one 8-day PACE image. All pixels are rerun here with the exact OCI band wavelengths.
- 10,000 PACE pixels drawn at random from the interiors of every eddy composite in the gold pigment table (March 2024 to May 2026), used for the spectra in section 1.
- The 145 SDP training samples (Kramer et al. 2022), each with a reflectance spectrum measured in the water and HPLC pigments from the same water.
- BATS HPLC pigments from the upper 10 m, 1989 to 2025, at 31.7°N 64.2°W in the Sargasso Sea (BCO-DMO dataset 893521).
- Ten PACE Level-2 granules over BATS from January to August 2025 (`data/sdp_validation/pace_l2/`).

How SDP works, in short. PACE measures the reflectance spectrum (Rrs), the share of sunlight leaving the ocean at each wavelength. SDP fits a smooth optical model to the spectrum, takes what the fit leaves over, and computes its second derivative on a 1 nm grid from 401 to 699 nm. The second derivative measures how sharply the curve bends at each wavelength: positive where the spectrum has a dip, negative where it has a bump. Each pigment is a weighted sum of the second derivative over all wavelengths, with weights fitted to the 145 training samples.

The training spectra were measured from ships, with no atmosphere between the instrument and the water. PACE looks through the whole atmosphere, and NASA's atmospheric correction removes the atmosphere's contribution before the reflectance reaches us. Anything the correction leaves behind is a feature SDP never saw in training.

Two features of sunlight and air matter below:
- Solar absorption lines, also called Fraunhofer lines: narrow dark lines in the sun's own spectrum, such as the G band at 430.8 nm and hydrogen beta (Hβ) at 486.1 nm.
- The oxygen B band: a narrow band from about 686 to 695 nm where oxygen in the air absorbs strongly.

The solar zenith angle is the angle between the sun and straight overhead. The larger it is, the longer the path sunlight takes through the air: at 60° the path is twice as long as with the sun overhead. At PACE's early-afternoon overpass it is about 22° in June and 59° in December for eddies south of the Gulf Stream, and a few degrees larger to the north.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import netCDF4 as nc
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

from eddy_tracking.packages.sdp.ancillary import sample_ancillary
from eddy_tracking.packages.sdp.physics import get_rrs_residuals
from eddy_tracking.packages.sdp.prediction import _load_coefficients
from eddy_tracking.packages.sdp.preprocessing import preprocess_rrs_batch
from eddy_tracking.preprocess.sss import read_multiple_sss
from eddy_tracking.preprocess.sst import read_multiple_sst

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
from train_sdp import train_rrs_model  # noqa: E402
from eddy_tracking.config import load_analysis_config
from eddy_tracking.utils.analysis_period import restrict_to_analysis_period

cfg = load_analysis_config("gulf_stream_20240305_20260531")

AUDIT = ROOT / "data" / "sdp_validation" / "gulf_stream_validity_20261013" / "envelope"
CACHE = ROOT / "data" / "sdp_validation" / "pace_spectra_eda_20261014"
L2_DIR = ROOT / "data" / "sdp_validation" / "pace_l2"
BATS_FILE = ROOT.parent / "sdp-validation" / "experiments" / "kramer_hplc_clustering" / "inputs" / "in_situ_hplc" / "BATS_HPLC_BCO-DMO-893521_v9_upper_10m.csv"
CACHE.mkdir(exist_ok=True)

arrays = np.load(AUDIT / "sample_arrays_main.npz")
pixels = pd.read_parquet(AUDIT / "sample_scored.parquet")
oci_wavelengths = arrays["wl_true"]
training_d2 = np.load(AUDIT / "train_d2.npz")["d2"]
training_hplc = pd.read_parquet(AUDIT / "train_hplc.parquet")
pigments = list(training_hplc.columns)

# The second derivative runs over 401 to 699 nm at 1 nm
d2_wavelengths = np.arange(401, 700)
training_mean = training_d2.mean(axis=0)
training_sd = training_d2.std(axis=0)

oci_gap = (588, 613)

side_names = {"N": "North of the Gulf Stream", "S": "South of the Gulf Stream"}
side_colors = {"N": "#3a6ea5", "S": "#c8553d"}
month_labels = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]

plt.rcParams.update({"font.size": 12, "axes.spines.top": False, "axes.spines.right": False})

## Rerun SDP with exact wavelengths

The cell below pushes every sampled pixel through the pipeline's own preprocessing and smooth-model fit with the exact OCI wavelengths, then applies the current SDP weights. It takes about 40 seconds the first time and caches the second derivative in `data/sdp_validation/pace_spectra_eda_20261014/`.

The solar zenith angle comes from each pixel's latitude and composite date at about 13:30 local solar time, PACE's overpass time. An 8-day composite mixes several overpasses, so this is an approximation.

In [ ]:
exact_path = CACHE / "exact_d2.npy"
if not exact_path.exists():
    wavelengths_1nm, smoothed = preprocess_rrs_batch(oci_wavelengths, arrays["raw"].astype(float))
    with np.errstate(invalid="ignore"):
        residual = get_rrs_residuals(pd.DataFrame(smoothed, columns=wavelengths_1nm.astype(int)), pixels["sst"].to_numpy(), pixels["sss"].to_numpy(), wavelengths_1nm)[1]
    np.save(exact_path, np.diff(np.asarray(residual), 2, axis=0).T)
d2 = np.load(exact_path)

current = {pigment: np.median(d2 @ _load_coefficients(pigment)[0] + _load_coefficients(pigment)[1], axis=1) for pigment in pigments}

day = pixels["date"].dt.dayofyear.to_numpy()
declination = np.radians(23.44 * np.sin(np.radians(360 / 365 * (day - 81))))
latitude = np.radians(pixels["pixel_lat"].to_numpy())
cos_zenith = np.sin(latitude) * np.sin(declination) + np.cos(latitude) * np.cos(declination) * np.cos(np.radians(22.5))
pixels["solar_zenith"] = np.degrees(np.arccos(cos_zenith))
pixels["composite"] = pixels.groupby(["track_id", "polarity", "date"]).ngroup()


def average_composites(tchla, zea=None):
    table = pixels[["composite", "side", "month", "chlor_a", "solar_zenith"]].assign(tchla=tchla, zea=zea if zea is not None else np.nan)
    composites = table.groupby("composite").agg(side=("side", "first"), month=("month", "first"), n=("tchla", "size"), chlor_a=("chlor_a", "mean"), solar_zenith=("solar_zenith", "mean"), tchla=("tchla", "mean"), zea=("zea", "mean"))
    # Pixel values stay signed so the mean is unbiased; a composite mean below zero is read as zero
    composites[["tchla", "zea"]] = composites[["tchla", "zea"]].clip(lower=0)
    return composites[composites["n"] >= 5]


bats = pd.read_csv(BATS_FILE, skiprows=[1])
bats["month"] = pd.to_datetime(bats["time"]).dt.month
# BATS pigment codes: p14 total chlorophyll a, p19 zeaxanthin, in ng/kg, so divide by 1000 for mg m⁻³
bats = bats.assign(tchla=pd.to_numeric(bats["p14"], errors="coerce") / 1000, zea=pd.to_numeric(bats["p19"], errors="coerce") / 1000)
bats = bats[bats["tchla"] > 0]
bats_monthly = bats.groupby("month").agg(tchla=("tchla", "median"), q25=("tchla", lambda s: s.quantile(0.25)), q75=("tchla", lambda s: s.quantile(0.75)))
bats_monthly["zea_ratio"] = (bats["zea"] / bats["tchla"]).groupby(bats["month"]).median()
print(f"[sdp] {len(pixels)} pixels, {pixels['composite'].nunique()} composites, {len(bats)} BATS samples from {bats['time'].min()[:4]} to {bats['time'].max()[:4]}")

## 1. PACE spectra carry solar-line and oxygen features that the training spectra lack

How to read the figure:
The PACE spectra here are not the audit sample used in the later sections. The cell below draws 10,000 pixels at random from the interiors of all gold-table eddy composites, with no balancing by polarity, side, or month, and reruns the pipeline's preprocessing and smooth-model fit on them with SST and SSS sampled the way `run_sdp` samples them. Each pixel is one cell of an 8-day PACE composite. Because every pixel has the same chance, large eddies, the south side (77% of the sample), and the clearer months of March to July (60%) carry more weight. The sample is cached in `data/sdp_validation/pace_spectra_eda_20261014/`.

How to read the figure:
- (a) The reflectance spectrum of the random PACE eddy pixels (black) and of the 145 SDP training spectra (orange). Lines are medians and the shading holds the middle 50% of spectra. PACE is drawn at the OCI band centers, the training spectra on SDP's 1 nm grid. OCI has no bands from 588 to 613 nm, so the PACE line is straight there.
- (b) PACE reflectance north and south of the Gulf Stream, drawn the same way.
- (c) The median second derivative of what is left after the smooth optical-model fit, on the 1 nm grid, which is what SDP's regression uses. Orange is the training spectra; blue and red are the PACE pixels north and south of the Gulf Stream.
- The strip above each wavelength axis names the rough color of light in each range.
- The table under the figure gives the G-band bump in single Level-2 overpasses over BATS: the mean reflectance at the two OCI bands inside the line minus the mean of the two bands beside it, as a percent of the reflectance beside the line and in absolute units, with the solar zenith angle and NASA's per-pixel Rrs uncertainty at 430 nm. Each row is the median of up to 3,000 clear pixels in one granule.

Outside the solar lines the PACE spectra fall inside the spread of the training spectra (a). At the median, PACE is brighter than the training set across the spectrum: 8.0 against 6.7 × 10⁻³ sr⁻¹ at 443 nm, and about twice as bright in the red (0.23 against 0.11 at 680 nm). South of the Gulf Stream the water is about 2.2 times brighter in the blue than north of it (9.2 against 4.2 at 443 nm), and above 520 nm the two sides match (b). The second derivative has nearly the same shape on both sides (c).

The largest departures from the training second derivative sit on solar lines (c). At 431 nm (the G band) the PACE median is about −19 × 10⁻⁶ sr⁻¹ nm⁻² north of the Gulf Stream and −27 south of it, while the training median is near zero; measured against the spread of the training second derivative, that is −5 to −7 standard deviations (SD) at 431 to 432 nm, −3 to −4 SD at 485 to 486 nm (Hβ), and smaller ones at 410, 517 to 518, and 527 nm. A negative second derivative is a bump, so PACE reflectance is slightly brighter inside these lines than beside them. The training spectra show no such structure even after blurring to OCI's band width. The bump is in single overpasses (table under the figure), so 8-day averaging does not create it. It is 1.3% to 2.2% of the reflectance, rising with the solar zenith angle (r = 0.96 over 10 granules), while its absolute size stays near 1.4 to 1.9 × 10⁻⁴ sr⁻¹, about a third of NASA's stated per-pixel Rrs uncertainty at 430 nm. Because the bump has the same sign in every pixel, averaging pixels does not remove it.

Partial filling of solar lines is a known effect of inelastic scattering, which shifts some light to nearby wavelengths: Raman scattering by water, and rotational Raman scattering by air (the Ring effect). Which one leaves this bump in PACE reflectance is not tested here.

North of the Gulf Stream the oxygen B band shows a small bump (+1 to +2 SD at 687 and 690 nm). It swings with the season (section 3) and is largest in winter, which this random sample holds few of.

In [ ]:
random_path = CACHE / "random_pixels.parquet"
random_d2_path = CACHE / "random_d2.npy"
if not random_path.exists():
    EXPERIMENT = ROOT / "data" / "gulf_stream_20240305_20260531"
    gold = restrict_to_analysis_period(pd.read_parquet(EXPERIMENT / "gold" / "eddy_pigment_table.parquet", columns=["track_id", "date", "polarity", "gs_dist_km"]), cfg)
    gold["polarity"] = gold["polarity"].map({0: "anticyclone", 1: "cyclone"})

    # Every interior pixel of every gold composite, then a simple random draw over that pool
    pool = []
    for (polarity, track_id), composites in gold.groupby(["polarity", "track_id"]):
        path = EXPERIMENT / "silver" / "collocate_pace" / str(polarity) / f"eddy_{track_id}_rrs.parquet"
        rows = pd.read_parquet(path, columns=["date", "inside_contour"])
        rows = rows[rows["inside_contour"] & rows["date"].isin(composites["date"])]
        pool.append(pd.DataFrame({"path": str(path), "row": rows.index, "polarity": polarity}))
    pool = pd.concat(pool, ignore_index=True)
    rng = np.random.default_rng(20261014)
    # Draw extra pixels because clouds leave some spectra incomplete
    drawn = pool.iloc[rng.choice(len(pool), 13000, replace=False)]

    parts = []
    for path, rows in drawn.groupby("path"):
        parts.append(pd.read_parquet(path).iloc[rows["row"]].assign(polarity=rows["polarity"].to_numpy()))
    sample = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=0)
    rrs_columns = [column for column in sample if column.startswith("Rrs_")]
    sample_wavelengths = np.array([float(column.split("_")[1]) for column in rrs_columns])
    needed = [column for column, wavelength in zip(rrs_columns, sample_wavelengths) if 390 < wavelength < 710]
    sample = sample[np.isfinite(sample[needed]).all(axis=1)]
    sample = sample.merge(gold, on=["polarity", "track_id", "date"])
    sample["side"] = np.where(sample["gs_dist_km"] > 0, "N", "S")

    # SST and SSS the way run_sdp samples them, reading only the files nearest each composite date
    sst_files = sorted((EXPERIMENT / "bronze" / "sst").glob("*.nc"))
    sss_files = sorted((EXPERIMENT / "bronze" / "sss").glob("*.nc4"))
    sst_times = pd.to_datetime([name.name.split(".")[1].split("_")[0] for name in sst_files]) + pd.Timedelta(days=3.5)
    sss_times = pd.to_datetime([name.name.split("_")[3] for name in sss_files])
    dates = pd.DatetimeIndex(sample["date"].unique())
    sst_grid = read_multiple_sst([sst_files[i] for i in np.unique(np.abs(sst_times.values[:, None] - dates.values[None, :]).argmin(axis=0))])
    sss_grid = read_multiple_sss([sss_files[i] for i in np.unique(np.abs(sss_times.values[:, None] - dates.values[None, :]).argmin(axis=0))])
    sample["sst"], sample["sss"] = sample_ancillary(sst_grid, sss_grid, sample["pixel_lon"].to_numpy(), sample["pixel_lat"].to_numpy(), sample["date"].to_numpy())
    sample = sample.dropna(subset=["sst", "sss"]).head(10000)

    wavelengths_1nm, smoothed = preprocess_rrs_batch(sample_wavelengths, sample[rrs_columns].to_numpy(float))
    with np.errstate(invalid="ignore"):
        residual = get_rrs_residuals(pd.DataFrame(smoothed, columns=wavelengths_1nm.astype(int)), sample["sst"].to_numpy(), sample["sss"].to_numpy(), wavelengths_1nm)[1]
    np.save(random_d2_path, np.diff(np.asarray(residual), 2, axis=0).T)
    sample.to_parquet(random_path)
random_pixels = pd.read_parquet(random_path)
random_d2 = np.load(random_d2_path)
print(f"[sdp] random sample: {len(random_pixels)} pixels from {random_pixels.groupby(['polarity', 'track_id', 'date']).ngroups} composites and {random_pixels.groupby(['polarity', 'track_id']).ngroups} eddies")


In [ ]:
l2_path = CACHE / "l2_g_band.parquet"
if not l2_path.exists():
    excluded = ["ATMFAIL", "LAND", "HIGLINT", "HILT", "HISATZEN", "STRAYLIGHT", "CLDICE", "COCCOLITH", "HISOLZEN", "LOWLW", "CHLFAIL", "NAVWARN", "MAXAERITER", "ATMWARN", "NAVFAIL"]
    rng = np.random.default_rng(0)
    rows = []
    for granule in sorted(L2_DIR.glob("*.nc")):
        with nc.Dataset(granule) as ds:
            geo = ds["geophysical_data"]
            flag_names = geo["l2_flags"].flag_meanings.split()
            bad = int(np.bitwise_or.reduce([int(geo["l2_flags"].flag_masks[flag_names.index(name)]) for name in excluded]))
            lat, lon = ds["navigation_data/latitude"][:], ds["navigation_data/longitude"][:]
            box = (lat > 28) & (lat < 36) & (lon > -70) & (lon < -58)
            row, col = np.where(box)
            r0, r1, c0, c1 = row.min(), row.max() + 1, col.min(), col.max() + 1
            clear = box[r0:r1, c0:c1] & ((geo["l2_flags"][r0:r1, c0:c1] & bad) == 0)
            ii, jj = np.where(clear)
            pick = rng.choice(len(ii), min(3000, len(ii)), replace=False)
            rrs = np.ma.filled(geo["Rrs"][r0:r1, c0:c1, :][ii[pick], jj[pick], :], np.nan)
            uncertainty = np.ma.filled(geo["Rrs_unc"][r0:r1, c0:c1, :][ii[pick], jj[pick], :], np.nan)
            band = {w: int(np.argmin(np.abs(geo["wavelength"][:] - w))) for w in [427.40, 429.89, 432.38, 434.87]}
            beside = (rrs[:, band[427.40]] + rrs[:, band[434.87]]) / 2
            bump = (rrs[:, band[429.89]] + rrs[:, band[432.38]]) / 2 - beside
            rows.append({"date": granule.name.split(".")[1][:8], "solar_zenith": float(np.mean(ds["scan_line_attributes/csol_z"][r0:r1][ii[pick]])), "bump_percent": 100 * float(np.nanmedian(bump / beside)), "bump": float(np.nanmedian(bump)), "uncertainty_430": float(np.nanmedian(uncertainty[:, band[429.89]]))})
    pd.DataFrame(rows).to_parquet(l2_path)
l2_bumps = pd.read_parquet(l2_path)

training_rrs = np.load(AUDIT / "train_d2.npz")["rrs"]
grid_1nm = np.arange(400, 701)
random_columns = [column for column in random_pixels.columns if column.startswith("Rrs_")]
random_wavelengths = np.array([float(column.split("_")[1]) for column in random_columns])
in_range = (random_wavelengths >= 400) & (random_wavelengths <= 700)
pace_rrs = random_pixels[random_columns].to_numpy(float)[:, in_range]
pace_wavelengths = random_wavelengths[in_range]

fig, axes = plt.subplots(3, 1, figsize=(12, 14), gridspec_kw={"hspace": 0.5})
ax = axes[0]
for rrs, wavelengths, color, label in [(training_rrs, grid_1nm, "#e09f3e", f"SDP training, measured in the water ({len(training_rrs)} spectra)"), (pace_rrs, pace_wavelengths, "black", f"PACE, eddy pixels ({len(pace_rrs):,} spectra)")]:
    q25, q50, q75 = np.nanpercentile(rrs, [25, 50, 75], axis=0)
    ax.fill_between(wavelengths, 1e3 * q25, 1e3 * q75, color=color, alpha=0.15, linewidth=0)
    ax.plot(wavelengths, 1e3 * q50, color=color, lw=2, label=label)
ax.set_ylim(0, 14)
ax.set_ylabel("Rrs (10⁻³ sr⁻¹)")
ax.set_title("a. Reflectance spectra, PACE and the SDP training data", loc="left")
ax.legend(frameon=False, loc="upper right")

ax = axes[1]
for side in ["N", "S"]:
    q25, q50, q75 = np.nanpercentile(pace_rrs[(random_pixels["side"] == side).to_numpy()], [25, 50, 75], axis=0)
    ax.fill_between(pace_wavelengths, 1e3 * q25, 1e3 * q75, color=side_colors[side], alpha=0.15, linewidth=0)
    ax.plot(pace_wavelengths, 1e3 * q50, color=side_colors[side], lw=2, label=side_names[side])
ax.set_ylim(0, 14)
ax.set_ylabel("Rrs (10⁻³ sr⁻¹)")
ax.set_title("b. PACE reflectance north and south of the Gulf Stream", loc="left")
ax.legend(frameon=False, loc="upper right")

ax = axes[2]
ax.plot(d2_wavelengths, 1e6 * np.median(training_d2, axis=0), color="#e09f3e", lw=1.5, label="SDP training")
for side, label in [("N", "PACE, north"), ("S", "PACE, south")]:
    ax.plot(d2_wavelengths, 1e6 * np.median(random_d2[(random_pixels["side"] == side).to_numpy()], axis=0), color=side_colors[side], lw=1.5, label=label)
ax.axhline(0, color="#bbbbbb", lw=0.8, zorder=0)
ax.set_ylim(-30, 30)
ax.set_ylabel("Second derivative (10⁻⁶ sr⁻¹ nm⁻²)")
ax.set_title("c. Second derivative of the residual spectrum, the input to SDP's regression", loc="left")
ax.legend(frameon=False, loc="lower right")

# Under each panel, a strip naming the rough color of light, carrying the wavelength axis
for ax in axes:
    ax.set_xlim(400, 700)
    ax.tick_params(axis="x", labelbottom=False, length=0)
    strip = ax.inset_axes([0, -0.09, 1, 0.09], sharex=ax)
    for lo, hi, color, name in [(400, 450, "#7b4fb3", "Violet"), (450, 495, "#3b6fd4", "Blue"), (495, 570, "#3fa34d", "Green"), (570, 590, "#e8c62e", "Yellow"), (590, 620, "#ee8a2a", "Orange"), (620, 700, "#d23c2f", "Red")]:
        strip.axvspan(lo, hi, color=color, linewidth=0)
        strip.text((lo + hi) / 2, 0.5, name, ha="center", va="center", fontsize=10, color="black" if name == "Yellow" else "white")
    strip.set_yticks([])
    strip.spines[["left", "bottom"]].set_visible(False)
    strip.set_xlabel("Wavelength (nm)")
fig.align_ylabels(axes)
plt.show()
l2_bumps.assign(bump=1e4 * l2_bumps["bump"], uncertainty_430=1e4 * l2_bumps["uncertainty_430"]).rename(columns={"bump": "bump (1e-4 sr⁻¹)", "uncertainty_430": "NASA uncertainty at 430 nm (1e-4 sr⁻¹)"}).round(2)

## 2. SDP Tchla follows `chlor_a` north of the Gulf Stream but not south of it

The figure shows monthly medians of eddy composites for (a) all eddies, (b) eddies north of the Gulf Stream, and (c) eddies south of it. Green is NASA's `chlor_a`, gray is SDP Tchla, and blue is Copernicus GlobColour CHL (CMEMS), all for the same composites. Copernicus CHL comes from the gold plankton table, whose 8-day dates match the PACE composites; each value there is the mean over the eddy's interior. Each composite value is the mean of its pixels, and the median is taken over the composites in that month. Panel (c) adds BATS HPLC Tchla in black, measured in the water near Bermuda from 1989 to 2025, with the shaded band holding the middle 50% of samples in each month; BATS sits in the Sargasso Sea, so it belongs only with the southern eddies. The y-axes differ because the north holds several times more chlorophyll.

SDP pixel values can be negative, which has no physical meaning. Each composite mean is clipped to zero after averaging, the rule the gold table follows, so a zero reads as "SDP finds no chlorophyll here". About a third of southern composites (32%) end up at zero, and 1% of northern ones.

North of the Gulf Stream, SDP Tchla follows `chlor_a` through the spring peak and the summer low, running 0.1 to 0.2 mg m⁻³ high in summer, and only breaks away in December and January (month-to-month r = 0.78, printed under the figure). South of it the two part ways (r = 0.40). BATS and `chlor_a` agree with each other: both peak in February and March at about 0.2 mg m⁻³ and fall to about 0.04 in July and August. SDP Tchla peaks in March and April, stays well above both through the summer, and is zero from October to December, when the real chlorophyll rises toward its winter high. Pooling both sides (a) blurs this, because the northern composites carry most of the chlorophyll. The sections below work backwards from the southern mismatch to its cause.

In [ ]:
current_composites = average_composites(current["Tchla"])
# Copernicus GlobColour CHL for the same eddy composites, from the gold plankton table on the same 8-day dates. The gold tables were rebuilt after this sample was drawn and track IDs changed, so each composite is matched to the Copernicus composite with the same polarity and date whose center is nearest, within 25 km
plankton_chl = restrict_to_analysis_period(pd.read_parquet(ROOT / "data" / "gulf_stream_20240305_20260531" / "gold" / "eddy_plankton_table.parquet", columns=["polarity", "date", "center_lat", "center_lon", "CHL"]), cfg)
composite_info = pixels.groupby("composite")[["polarity", "date", "center_lat", "center_lon"]].first().loc[current_composites.index]


def find_nearest_chl(row):
    candidates = plankton_chl[(plankton_chl["date"] == row.date) & (plankton_chl["polarity"] == row.polarity)]
    if candidates.empty:
        return np.nan
    km = np.hypot(111 * (candidates["center_lat"] - row.center_lat), 111 * np.cos(np.radians(row.center_lat)) * (candidates["center_lon"] - row.center_lon))
    return candidates["CHL"].iloc[int(km.argmin())] if km.min() <= 25 else np.nan


current_composites["cmems_chl"] = [find_nearest_chl(row) for row in composite_info.itertuples()]
north = current_composites[current_composites["side"] == "N"]
south = current_composites[current_composites["side"] == "S"]
monthly = south.groupby("month")[["chlor_a", "tchla", "cmems_chl"]].median()

fig, axes = plt.subplots(1, 3, figsize=(17, 5), gridspec_kw={"wspace": 0.3})
monthly_r = {}
for ax, (title, composites, top) in zip(axes, [("a. All eddies", current_composites, 0.6), ("b. North of the Gulf Stream", north, 1.0), ("c. South of the Gulf Stream", south, 0.35)]):
    by_month = composites.groupby("month")[["chlor_a", "tchla", "cmems_chl"]].median()
    monthly_r[title[3:]] = np.corrcoef(by_month["chlor_a"], by_month["tchla"])[0, 1]
    if composites is south:
        ax.fill_between(bats_monthly.index, bats_monthly["q25"], bats_monthly["q75"], color="black", alpha=0.1, linewidth=0)
        ax.plot(bats_monthly.index, bats_monthly["tchla"], color="black", lw=2.5, marker="o", label="BATS HPLC Tchla")
    ax.plot(by_month.index, by_month["chlor_a"], color="#4c956c", lw=2.5, marker="o", label="PACE chlor_a")
    ax.plot(by_month.index, by_month["tchla"], color="#7a7a7a", lw=2.5, marker="o", label="SDP Tchla")
    ax.plot(by_month.index, by_month["cmems_chl"], color="#3a6ea5", lw=2.5, marker="o", label="Copernicus CHL")
    ax.set_xticks(range(1, 13), month_labels)
    ax.set_xlim(0.5, 12.5)
    ax.set_ylim(0, top)
    ax.set_xlabel("Month")
    ax.set_ylabel("Chlorophyll a (mg m⁻³)")
    ax.set_title(title, loc="left")
    ax.legend(frameon=False, loc="upper right", fontsize=10)
plt.show()
print(f"[sdp] composites matched to Copernicus CHL within 25 km: {current_composites['cmems_chl'].notna().mean():.0%}; monthly r between Copernicus CHL and chlor_a, south {np.corrcoef(monthly['cmems_chl'], monthly['chlor_a'])[0, 1]:.2f}, with BATS Tchla {np.corrcoef(monthly['cmems_chl'], bats_monthly['tchla'])[0, 1]:.2f}")
print("[sdp] monthly r between SDP Tchla and chlor_a: " + ", ".join(f"{name} {r:.2f}" for name, r in monthly_r.items()) + f"; south SDP Tchla with BATS {np.corrcoef(monthly['tchla'], bats_monthly['tchla'])[0, 1]:.2f}")

## 3. One narrow band, 686 to 695 nm, sets the timing of the error

SDP adds up the second derivative at every wavelength times a weight. Of all wavelengths, 688 nm is the one whose part of that sum changes most over the year. It sits inside the oxygen B band (686 to 695 nm), where oxygen in the air absorbs sunlight strongly and NASA's atmospheric correction has to remove that absorption before the reflectance reaches us.

How to read the figure:
- (a) The PACE second derivative at 688 nm, monthly medians of eddy composites north (blue) and south (red) of the Gulf Stream. The gray band is the range of the 145 training spectra (mean ± 1 SD).
- (b) What the 686 to 695 nm band adds to SDP Tchla in each month: the second derivative at each of those wavelengths times SDP's Tchla weight, summed over the band.
- (c) The southern monthly cycle from section 2, plus SDP Tchla recomputed with the 686 to 695 nm part replaced by its average over all southern composites (purple), so that this band can no longer change from month to month. The band is swapped on the signed composite mean, and the result is then clipped to zero like every other composite mean.

In the training spectra the second derivative at 688 nm stays between about −1.7 and +0.6 × 10⁻⁶ sr⁻¹ nm⁻². PACE leaves that range in most months (a). It swings from about −6 in July to +5 in December south of the Gulf Stream, and from −4 to +16 north of it, so the two sides differ most in winter. SDP's weights at these wavelengths are negative, so the swing passes straight into Tchla with its sign flipped (b): the band adds about +0.2 mg m⁻³ in summer and takes away 0.3 in December south of the Gulf Stream and 0.9 north of it. That is more than all the chlorophyll in southern water for most of the year.

Holding this one band at its average puts SDP's seasonal cycle back in step with BATS (c): the month-to-month correlation rises from 0.18 to 0.88 (printed under the figure). The level is still off, because the recomputed Tchla falls to zero from June to September, so other wavelengths carry error as well. The timing of the seasonal error comes from this band.

In [ ]:
# The prediction is the median over 100 fitted models; their mean weights split it by wavelength closely enough
tchla_weights = _load_coefficients("Tchla")[0].mean(axis=1)
composite_d2 = pd.DataFrame(d2).groupby(pixels["composite"]).mean().loc[current_composites.index].to_numpy()
at_688 = d2_wavelengths == 688
oxygen = (d2_wavelengths >= 686) & (d2_wavelengths <= 695)
current_composites["d2_688"] = composite_d2[:, at_688][:, 0]
current_composites["oxygen_tchla"] = composite_d2[:, oxygen] @ tchla_weights[oxygen]
by_side = current_composites.groupby(["side", "month"])[["d2_688", "oxygen_tchla"]].median()
# Swap the band on the signed composite mean, then clip, since the parts only add up before clipping
south_oxygen = current_composites.loc[south.index, "oxygen_tchla"]
signed_tchla = pd.Series(current["Tchla"]).groupby(pixels["composite"]).mean().loc[south.index]
held = (signed_tchla - south_oxygen + south_oxygen.mean()).clip(lower=0).groupby(south["month"]).median()

fig, axes = plt.subplots(1, 3, figsize=(17, 5), gridspec_kw={"wspace": 0.3})
ax = axes[0]
ax.axhspan(1e6 * (training_mean[at_688][0] - training_sd[at_688][0]), 1e6 * (training_mean[at_688][0] + training_sd[at_688][0]), color="#7a7a7a", alpha=0.2, linewidth=0, label="Training spectra, mean ± 1 SD")
for side in ["N", "S"]:
    ax.plot(range(1, 13), 1e6 * by_side.xs(side)["d2_688"], color=side_colors[side], lw=2.5, marker="o", label=side_names[side])
ax.axhline(0, color="#bbbbbb", lw=0.8, zorder=0)
ax.set_xticks(range(1, 13), month_labels)
ax.set_xlim(0.5, 12.5)
ax.set_ylim(-10, 20)
ax.set_xlabel("Month")
ax.set_ylabel("Second derivative at 688 nm (10⁻⁶ sr⁻¹ nm⁻²)")
ax.set_title("a. PACE second derivative at 688 nm", loc="left")
ax.legend(frameon=False, loc="upper center", fontsize=10)

ax = axes[1]
for side in ["N", "S"]:
    ax.plot(range(1, 13), by_side.xs(side)["oxygen_tchla"], color=side_colors[side], lw=2.5, marker="o", label=side_names[side])
ax.axhline(0, color="#bbbbbb", lw=0.8, zorder=0)
ax.set_xticks(range(1, 13), month_labels)
ax.set_xlim(0.5, 12.5)
ax.set_ylim(-1, 0.4)
ax.set_xlabel("Month")
ax.set_ylabel("Part of SDP Tchla (mg m⁻³)")
ax.set_title("b. What 686 to 695 nm adds to SDP Tchla", loc="left")
ax.legend(frameon=False, loc="lower center", fontsize=10)

ax = axes[2]
ax.fill_between(bats_monthly.index, bats_monthly["q25"], bats_monthly["q75"], color="black", alpha=0.1, linewidth=0)
ax.plot(bats_monthly.index, bats_monthly["tchla"], color="black", lw=2.5, marker="o", label="BATS HPLC Tchla")
ax.plot(monthly.index, monthly["chlor_a"], color="#4c956c", lw=2.5, marker="o", label="PACE chlor_a")
ax.plot(monthly.index, monthly["tchla"], color="#7a7a7a", lw=2.5, marker="o", label="SDP Tchla")
ax.plot(held.index, held, color="#7b4fb3", lw=2.5, marker="o", label="SDP Tchla, 686–695 nm held at its average")
ax.axhline(0, color="#bbbbbb", lw=0.8, zorder=0)
ax.set_xticks(range(1, 13), month_labels)
ax.set_xlim(0.5, 12.5)
ax.set_ylim(0, 0.5)
ax.set_xlabel("Month")
ax.set_ylabel("Chlorophyll a (mg m⁻³)")
ax.set_title("c. South of the Gulf Stream, monthly medians", loc="left")
ax.legend(frameon=False, loc="upper right", fontsize=10)
plt.show()
print(f"[sdp] monthly r with BATS Tchla, south: SDP {np.corrcoef(monthly['tchla'], bats_monthly['tchla'])[0, 1]:.2f}, with 686–695 nm held at its average {np.corrcoef(held, bats_monthly['tchla'])[0, 1]:.2f}")

## 4. Why the band swings: the sun angle

The oxygen absorption that the atmospheric correction must remove depends on how much air the sunlight crosses. When the sun is low the path through the air is longer and the absorption deeper, so any error in removing it grows. The sun is lower in winter, and at any date it is lower to the north.

The figure plots the second derivative at 688 nm of each eddy composite against the solar zenith angle at PACE's early-afternoon overpass, computed from the composite's latitude and date. On both sides the second derivative rises with the sun angle (r = 0.77), slowly at first and steeply above about 60°, the way the path length grows. Where the two sides share a sun angle their points largely overlap, with the north a little higher. So most of the north-south difference in section 3 comes from the lower sun to the north. At a given angle the points still scatter by about ±5 × 10⁻⁶; chlorophyll fluorescence near 685 nm and the state of the atmosphere on the day are possible sources, not tested here.

The sun angle is the same for every pixel in a scene, so this error is shared by all the pixels of an eddy composite, and averaging over the eddy, as the gold table does, cannot remove it. Pixel-to-pixel differences inside a composite hold only 17% of the variance of SDP Tchla (2% for `chlor_a`). Across southern composites, `chlor_a` alone explains 34% of the variance of SDP Tchla, and `chlor_a` plus the solar zenith angle explains 66%. Both lines are printed under the figure.

In [ ]:
def compute_within_share(values):
    table = pd.DataFrame({"value": values, "composite": pixels["composite"]})
    table = table[table.groupby("composite")["value"].transform("size") >= 5]
    within = (table["value"] - table.groupby("composite")["value"].transform("mean")).pow(2).sum() / (len(table) - table["composite"].nunique())
    return within / table["value"].var()


fig, ax = plt.subplots(figsize=(8, 5.5))
ax.axhspan(1e6 * (training_mean[at_688][0] - training_sd[at_688][0]), 1e6 * (training_mean[at_688][0] + training_sd[at_688][0]), color="#7a7a7a", alpha=0.2, linewidth=0, label="Training spectra, mean ± 1 SD")
for side in ["N", "S"]:
    composites = current_composites[current_composites["side"] == side]
    r = np.corrcoef(composites["solar_zenith"], composites["d2_688"])[0, 1]
    ax.scatter(composites["solar_zenith"], 1e6 * composites["d2_688"], s=12, color=side_colors[side], alpha=0.4, linewidths=0, label=f"{side_names[side]}, r = {r:.2f}")
ax.axhline(0, color="#bbbbbb", lw=0.8, zorder=0)
ax.set_xlim(15, 75)
ax.set_ylim(-20, 40)
ax.set_xlabel("Solar zenith angle at the PACE overpass (°)")
ax.set_ylabel("Second derivative at 688 nm (10⁻⁶ sr⁻¹ nm⁻²)")
ax.set_title("Eddy composites: 688 nm against the sun angle", loc="left")
ax.legend(frameon=False, loc="upper left", fontsize=10, markerscale=2)
plt.show()

within = {pigment: compute_within_share(values) for pigment, values in current.items()}
print(f"[sdp] variance within composites: log chlor_a {compute_within_share(np.log10(pixels['chlor_a'].to_numpy())):.0%}, SDP Tchla {within['Tchla']:.0%}, all SDP pigments {min(within.values()):.0%} to {max(within.values()):.0%}")
design = np.column_stack([np.ones(len(south)), south["chlor_a"], south["solar_zenith"]])
fit = np.linalg.lstsq(design, south["tchla"].to_numpy(), rcond=None)[0]
r2_with_sun = 1 - np.var(south["tchla"].to_numpy() - design @ fit) / np.var(south["tchla"].to_numpy())
print(f"[sdp] south composites: chlor_a explains {np.corrcoef(south['tchla'], south['chlor_a'])[0, 1] ** 2:.0%} of SDP Tchla variance, chlor_a plus solar zenith angle {r2_with_sun:.0%}")

## 5. Retraining SDP without the affected wavelengths

The training spectra do not contain the oxygen band residual or the solar-line bumps, so the fairest repair is to fit SDP again on the 145 training samples with those wavelengths left out, using the repository's own training routine (`scripts/train_sdp.py`: 100 random 75/25 splits, up to 30 principal components chosen by 5-fold cross-validation). Five versions are fitted:
- All wavelengths. This reproduces the current weights (their Tchla weights correlate at 0.999), so it serves as the baseline.
- Without 685 to 699 nm, the oxygen band.
- Without 670 to 699 nm, the whole red end.
- Without 670 to 699 nm and without the solar-line windows (408 to 412, 428 to 440, 484 to 489, 515 to 520, and 525 to 529 nm).
- The same, also without 588 to 613 nm, where OCI has no bands.

Each version is scored in two ways. On the training data, by R² on the held-out 25% of samples, which says how much pigment information the remaining wavelengths carry in clean spectra. On PACE, by how well composite Tchla follows `chlor_a` on each side of the Gulf Stream, by how closely the monthly cycle south of the Gulf Stream follows BATS HPLC, and by the Zea / Tchla monthly cycle against BATS. Fitting all five takes about 9 minutes the first time and is cached.

The wavelengths left out were chosen after looking at PACE, by physical features named in advance (the oxygen band, known solar lines, the OCI gap), not by searching for the best score. BATS played no part in fitting.

In [ ]:
lines_mask = np.zeros(len(d2_wavelengths), dtype=bool)
for lo, hi in [(408, 412), (428, 440), (484, 489), (515, 520), (525, 529)]:
    lines_mask |= (d2_wavelengths >= lo) & (d2_wavelengths <= hi)
gap_mask = (d2_wavelengths >= oci_gap[0]) & (d2_wavelengths <= oci_gap[1])
versions = {
    "All wavelengths": d2_wavelengths <= 699,
    "Without 685–699 nm": d2_wavelengths <= 684,
    "Without 670–699 nm": d2_wavelengths <= 669,
    "Without 670–699 nm and solar lines": (d2_wavelengths <= 669) & ~lines_mask,
    "Without 670–699 nm, solar lines, and gap": (d2_wavelengths <= 669) & ~lines_mask & ~gap_mask,
}

retrained, skill = {}, []
for number, (version, keep) in enumerate(versions.items()):
    path = CACHE / f"retrained_{number}.npz"
    if not path.exists():
        fitted = {}
        # One thread: the training matrices are tiny, and BLAS threading makes the fit far slower. The routine divides by HPLC values that are zero, which only affects its percent-error summaries
        with threadpool_limits(1), np.errstate(divide="ignore", invalid="ignore"):
            for pigment in pigments:
                coefficients, intercepts, summary, _ = train_rrs_model(training_d2[:, keep], training_hplc[pigment].to_numpy(float), "pigment", 100, 30, 5, "MAE", seed=100)
                weights = np.zeros((len(d2_wavelengths), coefficients.shape[1]))
                weights[keep] = coefficients
                fitted[f"{pigment}_weights"], fitted[f"{pigment}_intercepts"], fitted[f"{pigment}_r2"] = weights, intercepts, summary["Mean_R2"].iloc[0]
        np.savez(path, **fitted)
    fitted = np.load(path)
    retrained[version] = {pigment: np.median(d2 @ fitted[f"{pigment}_weights"] + fitted[f"{pigment}_intercepts"], axis=1) for pigment in pigments}
    skill.append({"version": version, **{pigment: float(fitted[f"{pigment}_r2"]) for pigment in pigments}})
skill = pd.DataFrame(skill).set_index("version")

scores = []
for version, predicted in {"Current weights": current, **retrained}.items():
    composites = average_composites(predicted["Tchla"], predicted["Zea"])
    north_composites, south_composites = composites[composites["side"] == "N"], composites[composites["side"] == "S"]
    monthly_tchla = south_composites.groupby("month")["tchla"].median()
    # Zea / Tchla only where composite Tchla is above 0.02 mg m⁻³, so near-zero Tchla does not blow up the ratio
    zea_ratio = (south_composites["zea"].clip(lower=0) / south_composites["tchla"]).where(south_composites["tchla"] > 0.02).groupby(south_composites["month"]).median()
    scores.append({
        "version": version,
        "training R² Tchla": skill.loc[version, "Tchla"] if version in skill.index else np.nan,
        "r with chlor_a, north": north_composites[["tchla", "chlor_a"]].corr().iloc[0, 1],
        "r with chlor_a, south": south_composites[["tchla", "chlor_a"]].corr().iloc[0, 1],
        "south composites ≤ 0 (%)": 100 * (south_composites["tchla"] <= 0).mean(),
        "south Jun–Sep Tchla − chlor_a": (south_composites["tchla"] - south_composites["chlor_a"])[south_composites["month"].between(6, 9)].median(),
        "monthly r with BATS Tchla": np.corrcoef(monthly_tchla, bats_monthly["tchla"])[0, 1],
        "monthly r with BATS Zea / Tchla": np.corrcoef(zea_ratio.reindex(range(1, 13)).fillna(zea_ratio.mean()), bats_monthly["zea_ratio"])[0, 1],
    })
scores = pd.DataFrame(scores).set_index("version")
scores.round(2)

The table compares the versions. Leaving out the red end and the solar lines costs some skill on the clean training spectra: held-out R² for Tchla falls from 0.73 to 0.60, mostly because 670 to 684 nm carries chlorophyll's red absorption peak. On PACE the same versions do much better:
- Across eddy composites south of the Gulf Stream, the correlation of SDP Tchla with `chlor_a` rises from 0.58 with the current weights to 0.84 to 0.90 once the oxygen band is left out, and north of it from 0.81 to 0.82 to 0.85.
- The monthly cycle south of the Gulf Stream follows BATS at r = 0.66 to 0.86 depending on the version, against 0.18 for the current weights, and the Zea / Tchla cycle follows BATS at 0.68 to 0.75, against 0.20.
- Keeping 670 to 684 nm (the version without only 685 to 699 nm) matches the BATS seasonal shape best but puts 49% of southern composites at zero.

The figure below uses the version without 670 to 699 nm and solar lines, which gives the highest correlation with `chlor_a` on both sides.
- (a) Held-out R² on the 145 training samples for each pigment, current weights (gray) against the retrained version (red).
- (b) The monthly cycle south of the Gulf Stream, as in section 2, with the retrained version added.
- (c) Each dot is one eddy composite south of the Gulf Stream: `chlor_a` against SDP Tchla from the current weights (gray) and the retrained version (red). The dotted line is Tchla = `chlor_a`.

In [ ]:
chosen = "Without 670–699 nm and solar lines"
chosen_composites = average_composites(retrained[chosen]["Tchla"])
chosen_south = chosen_composites[chosen_composites["side"] == "S"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), gridspec_kw={"width_ratios": [1, 1.1, 1], "wspace": 0.32})
ax = axes[0]
skill_by_version = skill.T
order = skill_by_version["All wavelengths"].sort_values().index
positions = np.arange(len(order))
ax.hlines(positions, skill_by_version.loc[order, chosen], skill_by_version.loc[order, "All wavelengths"], color="#bbbbbb", lw=1.5)
ax.plot(skill_by_version.loc[order, "All wavelengths"], positions, "o", color="#7a7a7a", label="All wavelengths")
ax.plot(skill_by_version.loc[order, chosen], positions, "o", color="#c8553d", label="Retrained")
ax.set_yticks(positions, order)
ax.set_xlim(0.2, 0.8)
ax.set_xlabel("Held-out R² on training samples")
ax.set_title("a. Skill on in-water spectra", loc="left")
ax.legend(frameon=False, loc="lower right", fontsize=10)

ax = axes[1]
ax.fill_between(bats_monthly.index, bats_monthly["q25"], bats_monthly["q75"], color="black", alpha=0.1, linewidth=0)
ax.plot(bats_monthly.index, bats_monthly["tchla"], color="black", lw=2.5, marker="o", label="BATS HPLC Tchla")
ax.plot(monthly.index, monthly["chlor_a"], color="#4c956c", lw=2.5, marker="o", label="PACE chlor_a")
ax.plot(monthly.index, monthly["tchla"], color="#7a7a7a", lw=2, ls="--", marker="o", label="SDP Tchla, current weights")
ax.plot(monthly.index, chosen_south.groupby("month")["tchla"].median(), color="#c8553d", lw=2.5, marker="o", label="SDP Tchla, retrained")
ax.axhline(0, color="#bbbbbb", lw=0.8, zorder=0)
ax.set_xticks(range(1, 13), month_labels)
ax.set_xlim(0.5, 12.5)
ax.set_ylim(0, 0.4)
ax.set_ylabel("Median (mg m⁻³)")
ax.set_title("b. South of the Gulf Stream, by month", loc="left")
ax.legend(frameon=False, loc="upper right", fontsize=10)

ax = axes[2]
r_current = south[["tchla", "chlor_a"]].corr().iloc[0, 1]
r_chosen = chosen_south[["tchla", "chlor_a"]].corr().iloc[0, 1]
ax.scatter(south["chlor_a"], south["tchla"], s=10, color="#7a7a7a", alpha=0.5, linewidths=0, label=f"Current weights, r = {r_current:.2f}")
ax.scatter(chosen_south["chlor_a"], chosen_south["tchla"], s=10, color="#c8553d", alpha=0.6, linewidths=0, label=f"Retrained, r = {r_chosen:.2f}")
one_to_one = np.linspace(0, 0.6, 50)
ax.plot(one_to_one, one_to_one, color="black", lw=1, ls=":")
ax.axhline(0, color="#bbbbbb", lw=0.8, zorder=0)
ax.set_xlim(0, 0.6)
ax.set_ylim(0, 1.0)
ax.set_xlabel("PACE chlor_a (mg m⁻³)")
ax.set_ylabel("SDP Tchla (mg m⁻³)")
ax.set_title("c. Eddy composites, south", loc="left")
ax.legend(frameon=False, loc="upper left", fontsize=10)
plt.show()

## How the pieces fit together

1. PACE reflectance contains two features that the in-water training spectra do not: small bumps on solar absorption lines (about 2% of the reflectance at the G band), and a residual at the oxygen B band near 687 nm. Both come from looking through the atmosphere, or from how the atmospheric correction handles it, so the training data never taught SDP what to do with them.
2. SDP's weights use these wavelengths. The red end, 670 to 699 nm, carries about a quarter of the Tchla weight, and its oxygen-band residual grows with the sun's path through the air. In clear Sargasso water, where the real chlorophyll signal is small, this turns SDP Tchla into a measure of the season's sun angle, with a cycle that runs against BATS HPLC and NASA's `chlor_a`.
3. These errors are shared by every pixel in a scene, so averaging over an eddy does not remove them.
4. Retraining SDP on the same 145 samples without the red end and the solar lines loses some skill on clean spectra but restores agreement with `chlor_a` (r = 0.90 south, 0.85 north across eddy composites) and gives a seasonal cycle and a Zea / Tchla cycle that follow BATS.

What still does not work:
- The level in clear summer water. The retrained Tchla is zero in about a third of southern composites (32%), and from June to September it sits about 0.03 mg m⁻³ below `chlor_a` at the median. SDP's own error on the training data (mean absolute error 0.16 to 0.19 mg m⁻³ for Tchla) is larger than all the chlorophyll in summer Sargasso water (about 0.04 mg m⁻³), so absolute Tchla levels and pigment ratios there stay out of reach with 145 training samples.
- The test is indirect. No HPLC sample here comes from an eddy at the time of a PACE image, BATS is one site on the south side, and `chlor_a` is a second satellite estimate, not a measurement.

Next tests:
- Match the 2025 PVST HPLC casts at BATS (`data/sdp_validation/pvst_bats_hplc/`) to the Level-2 granules above and score the current and retrained weights pixel by pixel.
- Look for the solar-line bump in the in-water reflectance that PVST collected near BATS (SeaBASS PVST_PSST). If it is there, the bump is ocean Raman scattering and the training spectra are too blurred to show it; if not, it comes from the atmospheric correction.
- Rerun the full pipeline with the retrained weights on Phoenix and check whether the eddy pigment trends in `research_paper/` change.